# Day 61 - quantization

A 70B model in bf16 is 140 GB, and every decode step reads all of it. Quantization is
the decision to store each weight in 8 bits, or 4, and the subject is entirely about
what the rounding does to the layer's output.

Three decisions separate every method here: which numbers share a scale, where the
outliers go, and whether the rounding minimises the error in the weights or in the
output.

In [ ]:
import numpy as np
from quantization import (make_layer, quant_int, best_clip, quant_fp8, mxfp4,
                          float_round, format_table, dynamic_range, where_the_error_is,
                          smooth_quant, smooth_sweep, mixed_precision, awq_scale, gptq,
                          rounding_table, qat_demo, memory_table, out_err, FP8,
                          format_spec_table, mx_table, bits_of_float32, spec)

L = make_layer()
W, X = L['W'], L['X']
Y = X @ W.T
for r in memory_table():
    print(f"{r['name']:18} {r['bits']:5.2f} bits   {r['gb']:6.1f} GB   {r['ms_at_5TBs']:5.1f} ms per read")

bf16               16.00 bits    140.0 GB    28.0 ms per read
fp8                 8.00 bits     70.0 GB    14.0 ms per read
int8                8.00 bits     70.0 GB    14.0 ms per read
int4 + group 128    4.12 bits     36.1 GB     7.2 ms per read
mxfp4               4.25 bits     37.2 GB     7.4 ms per read


## 1. Every format on one page

A float is three fields in one word - sign, exponent, mantissa - read as
`(-1)^sign * 2^(exponent - bias) * 1.mantissa`. The exponent is stored biased so that
floats compare correctly as integers, and the leading 1 is implied, which is one free
bit of precision.

In [ ]:
b = bits_of_float32(0.15625)
print(b['bits'], '=', b['sign'], b['exp_bits'], b['man_bits'])
print(f"({-1 if b['sign']=='1' else 1}) * 2^({b['stored_exp']} - {b['bias']}) * {b['mantissa']} = {b['check']}")

print()
print(f"{'format':12} {'bits':>4} {'S/E/M':>9} {'bias':>5} {'max':>11} {'rel step':>10}  used for")
for r in format_spec_table():
    sem = f"1/{r['e']}/{r['m']}" if r['e'] else f"1/-/{r['m']}"
    print(f"{r['name']:12} {r['bits']:4d} {sem:>9} {str(r['bias']):>5} {r['max']:11.4g} {r['eps']:10.3g}  {r['use']}")

print()
for r in mx_table():
    print(f"{r['name']:12} element {r['element_bits']}b + E8M0 per {r['block']} = {r['bits_per_value']:.2f} bits/value, element max {r['max_element']:.0f}")

00111110001000000000000000000000 = 0 01111100 01000000000000000000000
(1) * 2^(124 - 127) * 1.25 = 0.15625

format       bits     S/E/M  bias         max   rel step  used for
fp32           32    1/8/23   127   3.403e+38   1.19e-07  the default everywhere
tf32           19    1/8/10   127   3.401e+38   0.000977  NVIDIA tensor cores: fp32 range, fp16 precision
bf16           16     1/8/7   127    3.39e+38    0.00781  training: fp32 range, half the bytes
fp16           16    1/5/10    15    6.55e+04   0.000977  older training, needs loss scaling
fp8 e4m3fn      8     1/4/3     7         448      0.125  FP8 forward pass: weights and activations
fp8 e5m2        8     1/5/2    15   5.734e+04       0.25  FP8 gradients: more range, less precision
fp6 e3m2        6     1/3/2     3          28       0.25  MX block element, more range
fp6 e2m3        6     1/2/3     1         7.5      0.125  MX block element, more precision
fp4 e2m1        4     1/2/1     1           6        0.5  MXFP4 / NVFP4 

## 3. What a quantized number is, and who shares a scale

In [ ]:
v = np.array([0.031, -0.0072, 0.0009, 0.118])
s = np.abs(v).max() / 127
for x in v:
    print(f"{x:+.4f} -> q={int(round(x/s)):4d} -> {round(x/s)*s:+.6f}  (error {abs(round(x/s)*s-x):.6f})")

print()
for r in format_table(W):
    print(f"{r['name']:20} {r['bits']:5.2f} bits   rel err {r['rel']:.5f}")

+0.0310 -> q=  33 -> +0.030661  (error 0.000339)
-0.0072 -> q=  -8 -> -0.007433  (error 0.000233)
+0.0009 -> q=   1 -> +0.000929  (error 0.000029)
+0.1180 -> q= 127 -> +0.118000  (error 0.000000)

int8, per tensor      8.00 bits   rel err 0.03103
int8, per row         8.00 bits   rel err 0.01444
fp8 e4m3              8.00 bits   rel err 0.02658
fp8 e5m2              8.00 bits   rel err 0.05281
int4, per row         4.00 bits   rel err 0.26197
int4, group 128       4.12 bits   rel err 0.17584
mxfp4, block 32       4.25 bits   rel err 0.12301


## 3. Integers against floats at the same width

FP8 spends bits on an exponent, so the spacing grows with the magnitude and the
*relative* error stays roughly constant. The E2M1 grid used by MXFP4 has eight
magnitudes.

In [ ]:
print(dynamic_range())
print('e2m1 grid:', sorted(set(np.abs(float_round(np.linspace(-6, 6, 20001), 1, 0, 2)))))
print('fp8 formats (mantissa bits, emin, emax, max):', FP8)

{'int8_steps': 254, 'int8_ratio': 127.0, 'e4m3_ratio': 229376.0, 'e5m2_ratio': 3758096384.0, 'fp4_levels': 15}
e2m1 grid: [np.float64(0.0), np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(6.0)]
fp8 formats (mantissa bits, emin, emax, max): {'e4m3': (3, -6, 8, 448.0), 'e5m2': (2, -14, 15, 57344.0)}


## 4. The outliers live in the activations

In [ ]:
for r in where_the_error_is(L):
    print(f"{r['name']:32} {r['rel']:.5f}")
print()
print('mixed precision:', mixed_precision(L))
for r in smooth_sweep(L):
    print(f"alpha {r['alpha']:.2f}  act range {r['act_range_after']:6.1f}  error {r['smooth']:.5f}")

weights only, per row            0.01462
activations only, per tensor     0.06897
activations only, per token      0.03394
both                             0.03703

mixed precision: {'kept': 6, 'share': 0.01171875, 'rel': 0.006861649913182409, 'all_int8': 0.03703201469377447}
alpha 0.00  act range   43.9  error 0.05468
alpha 0.25  act range   17.1  error 0.02243
alpha 0.50  act range    6.6  error 0.01348
alpha 0.75  act range    2.6  error 0.01925
alpha 1.00  act range    1.0  error 0.04043


## 5. Rounding better: GPTQ and AWQ

GPTQ ends up with worse weights and a better output, which is the point.

In [ ]:
rt = rounding_table(L, bits=4, group=128)
for k, v2 in rt.items():
    print(k, round(v2, 5) if isinstance(v2, float) else v2)

rtn_weight_err 0.17584
gptq_weight_err 0.31186
rtn_out 0.17612
gptq_out 0.13051
awq_out 0.10059
awq_alpha 0.36842
bits 4
group 128


## 6. Quantization-aware training, and where its gain comes from

In [ ]:
q = qat_demo()
qo = qat_demo(correlated=False)
print(f"correlated inputs: PTQ {q['ptq']:.3f} -> QAT {q['qat']:.3f}  ({q['ratio']:.2f}x)")
print(f"orthogonal inputs: PTQ {qo['ptq']:.3f} -> QAT {qo['qat']:.3f}  ({qo['ratio']:.2f}x)")
print(f"last iterate {q['qat_last']:.3f}, best {q['qat']:.3f}, swing +-{q['osc']:.2f}")

correlated inputs: PTQ 2.180 -> QAT 0.837  (2.60x)
orthogonal inputs: PTQ 1.632 -> QAT 1.474  (1.11x)
last iterate 1.226, best 0.837, swing +-1.17


## 7. Tests

In [ ]:
assert np.linalg.norm(quant_int(W, 8, axis=-1) - W) <= np.linalg.norm(quant_int(W, 8) - W)
assert np.linalg.norm(mxfp4(W) - W) < np.linalg.norm(quant_int(W, 4, axis=-1) - W)
assert best_clip(X, 8)['fraction'] < 1.0
sq = smooth_quant(L, 0.5)
assert sq['smooth'] < sq['plain']
rt = rounding_table(L, bits=4, group=128)
assert rt['gptq_out'] < rt['rtn_out'] and rt['gptq_weight_err'] > rt['rtn_weight_err']
print('all assertions passed')

all assertions passed
